# Phase 6: Summarisation: Encoder-decoder (6x6)

This is the **Summarisation** part of the full `encdec` notebook (`notebooks/CS781_encdec_all_phases.ipynb`), extracted so it can be read on its own. It reuses the tokenizer, model classes and helpers defined in the earlier cells, so **to run it, execute the full notebook from the top** (or run the Phase 1 set-up cells first, then this part).

## Abstractive Summarization


In [ ]:
import json, os, numpy as np
from huggingface_hub import snapshot_download
P6_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_6"; os.makedirs(P6_DIR, exist_ok=True)
local = snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                          allow_patterns=["Phase_6_Summ/*", "*/Phase_6_Summ/*"], token=HF_TOKEN)
def find(name):
    for r, _, fs in os.walk(local):
        if name in fs and "Phase_6" in r: return os.path.join(r, name)
def rows(d):
    if isinstance(d, list): return d
    if all(isinstance(v, dict) for v in d.values()): return [dict(id=k, **v) for k, v in d.items()]
    return [dict(zip(d.keys(), v)) for v in zip(*d.values())]
tr6, te6 = rows(json.load(open(find("train.json")))), rows(json.load(open(find("test.json"))))
print("train:", len(tr6), "| test:", len(te6), "| train keys:", list(tr6[0].keys()), "| test keys:", list(te6[0].keys()))
dk = "document" if "document" in tr6[0] else "text"
print("\nDOC start :", tr6[0][dk][:300].replace("\n", " "))
print("SUMMARY   :", tr6[0]["summary"][:300].replace("\n", " "))
print("duplicate ids:", len(tr6) - len({r["id"] for r in tr6}), "| empty summaries:", sum(not r["summary"].strip() for r in tr6))

def toklen(texts):
    return np.array([len(e.ids) for e in tokenizer.encode_batch([t[:300000] for t in texts])])
rng = np.random.RandomState(0); samp = [tr6[i] for i in rng.permutation(len(tr6))[:500]]
dl, sl = toklen([r[dk] for r in samp]), toklen([r["summary"] for r in samp])
tl = toklen([r[dk if dk in r else "document"] for r in [te6[i] for i in rng.permutation(len(te6))[:500]]])
q = lambda a: {p: int(np.percentile(a, p)) for p in (10, 50, 90, 99)} | {"max": int(a.max())}
print("\ndocument tokens (500 train docs):", q(dl))
print("summary  tokens (500 train docs):", q(sl))
print("test document tokens (500 docs) :", q(tl))
print("compression ratio (median doc/summary): %.1fx" % np.median(dl / np.maximum(sl, 1)))
for L in (1024, 2048, 3072, 4096): print(f"docs <= {L} tokens: {100 * (dl <= L).mean():.1f}%")
for L in (128, 256, 320, 512): print(f"summaries <= {L} tokens: {100 * (sl <= L).mean():.1f}%")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

train: 7030 | test: 100 | train keys: ['id', 'document', 'summary'] | test keys: ['id', 'document']

DOC start : IVIL Appeal No.768 (NT) of 1977 etc. From the judgment Order dated 9.10.1975 of the Madhya Pradesh High Court in M.C.C. No.144 of 1966. V.A. Bobde, B.R.Aggarwala and U.A.Rana for the Appellant. S.V.Deshpande and S.K.Agnihotri for the Respondents. The Judgment of the Court was delivered by RANGANATH 
SUMMARY   : The appellant, a manufacturer of cement, entered into an agreement with the Cement Manufacturing Company of India Limited, for sale of cement. Under the agreement, the appellant was to sell its cement only through the marketing company, and certain sums would be paid for the cement supplied by the m
duplicate ids: 0 | empty summaries: 2

document tokens (500 train docs): {10: 1673, 50: 3699, 90: 9867, 99: 34721, 'max': 51763}
summary  tokens (500 train docs): {10: 352, 50: 773, 90: 1833, 99: 5569, 'max': 9803}
test document tokens (500 docs) : {10: 1766, 50: 3754, 90:

In [16]:
import os, json, pickle, numpy as np
from huggingface_hub import snapshot_download
P6_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_6"; os.makedirs(P6_DIR, exist_ok=True)
CACHE6 = f"{P6_DIR}/sum_tok_cache.pkl"
EOS6 = tokenizer.token_to_id("<|endoftext|>")

def doc_view(rec, n_head, n_tail=0):
    """Token view of a document: first n_head tokens, plus last n_tail tokens if the doc is longer."""
    n, h, t = rec["n"], rec["head"], rec["tail"]
    full = h if n <= 4096 else (np.concatenate([h, t[-(n - 4096):]]) if n <= 6144 else None)
    if full is not None:
        return full if n <= n_head + n_tail else np.concatenate([full[:n_head], full[n - n_tail:]] if n_tail else [full[:n_head]])
    return np.concatenate([h[:n_head], t[-n_tail:]] if n_tail else [h[:n_head]])

if os.path.exists(CACHE6):
    D6 = pickle.load(open(CACHE6, "rb")); print("loaded cache from Drive")
else:
    local = snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                              allow_patterns=["Phase_6_Summ/*", "*/Phase_6_Summ/*"], token=HF_TOKEN)
    def find(name):
        for r, _, fs in os.walk(local):
            if name in fs and "Phase_6" in r: return os.path.join(r, name)
    tr = [r for r in json.load(open(find("train.json"))) if r["summary"].strip()]
    te = json.load(open(find("test.json")))
    def enc(texts, bs=128):
        out = []
        for i in range(0, len(texts), bs):
            out += [e.ids for e in tokenizer.encode_batch([t[:400000] for t in texts[i:i + bs]])]
        return out
    def pack(ids): return dict(n=len(ids), head=np.array(ids[:4096], np.uint16), tail=np.array(ids[-2048:], np.uint16))
    tr_docs, tr_sums = enc([r["document"] for r in tr]), enc([r["summary"] for r in tr])
    te_docs = enc([r["document"] for r in te])
    recs = [dict(id=r["id"], **pack(d), s_len=len(s), summ=np.array(s[:1024], np.uint16)) for r, d, s in zip(tr, tr_docs, tr_sums)]
    perm = np.random.RandomState(0).permutation(len(recs))
    D6 = dict(val=[recs[i] for i in perm[:300]], train=[recs[i] for i in perm[300:]],
              test=[dict(id=r["id"], **pack(d)) for r, d in zip(te, te_docs)])
    pickle.dump(D6, open(CACHE6, "wb")); print("saved cache to Drive")

print("train:", len(D6["train"]), "| val:", len(D6["val"]), "| test:", len(D6["test"]), "| EOS id:", EOS6)
r = D6["train"][0]
print("view check: head-only 1536 ->", len(doc_view(r, 1536)), "| head 1024 + tail 512 ->", len(doc_view(r, 1024, 512)), "| doc length", r["n"])

loaded cache from Drive
train: 6728 | val: 300 | test: 100 | EOS id: 0
view check: head-only 1536 -> 1536 | head 1024 + tail 512 -> 1536 | doc length 3103


In [11]:
import math, os, torch, torch.nn as nn, torch.nn.functional as F
KIND = "encdec"          # <-- set per notebook: gqa | strided | mla | encdec | student
assert "RRModel" in globals() and "tokenizer" in globals() and "device" in globals(), "run setup + Phase-3 toolkit first"
P1 = DRIVE_PROJECT_DIR                                    # Phase-1 folder (from the setup cell)
STAGE1_CKPT = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_2/student_kd_stage1/best.pt"

def _load(lm, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = ck.get("model", ck.get("model_state_dict", ck)) if isinstance(ck, dict) else ck
    lm.load_state_dict({k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()})
    return lm

def build_lm(kind):
    V = tokenizer.get_vocab_size()
    if kind == "gqa":
        return _load(CustomDecoderLM(vocab_size=V, d_model=768, n_layer=10, n_head=12, n_kv_head=4, max_seq_len=2048, tie_weights=True),
                     f"{P1}/gqa_checkpoints/gqa_best.pt"), 768
    if kind == "strided":
        pat = alternating_patterns(10, [(256, 32), (256, 128)])
        return _load(CustomDecoderLM(vocab_size=V, d_model=768, n_layer=10, n_head=12, n_kv_head=4, max_seq_len=2048,
                                     tie_weights=True, layer_patterns=pat), f"{P1}/alt_stride_32_128_checkpoints/best.pt"), 768
    if kind == "mla":
        return _load(CustomDecoderLM(vocab_size=V, d_model=768, n_layer=10, n_head=12, max_seq_len=4096, qk_rope_head_dim=64,
                                     tie_weights=True), f"{P1}/DeepSeek/deepseek_mla_85m_pretrained.pt"), 768
    if kind == "encdec":
        return _load(EncoderDecoderLM(vocab_size=V, d_model=640, n_enc_layer=6, n_dec_layer=6, n_head=10, n_kv_head=5,
                                      max_src_len=1024, max_tgt_len=320, bos_id=tokenizer.token_to_id("<|endoftext|>"),
                                      tie_weights=True), f"{P1}/encdec_6x6_checkpoints/best.pt"), 640
    if kind == "student":
        return _load(CustomDecoderLM(vocab_size=V, d_model=576, n_layer=7, n_head=12, n_kv_head=4, max_seq_len=2048,
                                     tie_weights=True), STAGE1_CKPT), 576
    raise ValueError(kind)

class Backbone(RRModel):
    """Phase-1 LM + a features() method. Task heads are attached by each phase (model.head = ...)."""
    def __init__(self, lm, d, kind):
        super().__init__(lm, d, 2, 0.1); del self.head
        self.kind, self.d = kind, d
    def features(self, ids, lens):                       # ids (chunks, T) right-padded, lens (chunks,) -> (chunks, 2d)
        if self.kind == "encdec":                        # encoder takes no mask: one chunk at a time
            out = []
            for j in range(ids.shape[0]):
                n = int(lens[j]); H = self.lm.encode(ids[j:j + 1, :n])[0].float()
                out.append(torch.cat([H[n - 1], H.mean(0)], -1))
            return torch.stack(out)
        H = self.hidden(ids); B, T, _ = H.shape
        mask = (torch.arange(T, device=H.device)[None] < lens[:, None]).unsqueeze(-1)
        return torch.cat([H[torch.arange(B, device=H.device), lens - 1].float(), (H.float() * mask).sum(1) / lens[:, None]], -1)

def make_backbone(kind=None):
    lm, d = build_lm(kind or KIND); return Backbone(lm, d, kind or KIND).to(device)

# ---------- adapters (own implementations) ----------
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__(); self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x): return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

class LoRMAPlusLinear(nn.Module):
    """W = ((alpha/r) B A + I) W0 on the output side. Identity at init, mergeable."""
    def __init__(self, base, r, alpha):
        super().__init__(); self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x); return y + self.scale * F.linear(F.linear(y, self.A), self.B)

TARGETS = {k: ("q_proj", "k_proj", "v_proj", "out_proj") for k in ["gqa", "strided", "encdec", "student"]}
TARGETS["mla"] = ("q_up_proj", "kv_down_proj", "kv_up_proj", "out_proj")
def layers_of(bb): return bb.lm.enc_layers if bb.kind == "encdec" else bb.lm.layers

def inject(bb, method, r, alpha):
    """Freeze the Phase-1 LM and add adapters. Heads attached outside bb.lm stay trainable."""
    for p in bb.lm.parameters(): p.requires_grad_(False)
    cls = {"lora": LoRALinear, "lorma": LoRMAPlusLinear}[method]
    for layer in layers_of(bb):
        for nm in TARGETS[bb.kind]:
            base = getattr(layer.attn, nm); setattr(layer.attn, nm, cls(base, r, alpha).to(base.weight.device))
    return bb

def unfreeze_top(bb, k):
    """Partial (layer-wise) fine-tuning: only the top k layers (+ final norm for decoders) train."""
    for p in bb.lm.parameters(): p.requires_grad_(False)
    for layer in list(layers_of(bb))[-k:]:
        for p in layer.parameters(): p.requires_grad_(True)
    if bb.kind != "encdec":
        for p in bb.lm.norm.parameters(): p.requires_grad_(True)
    return bb

def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad), sum(p.numel() for p in m.parameters())

# ---------- distillation losses ----------
def kd_multiclass(s, t, T=2.0): return F.kl_div(F.log_softmax(s / T, -1), F.softmax(t / T, -1), reduction="batchmean") * T * T
def kd_multilabel(s, t, T=2.0): return F.binary_cross_entropy_with_logits(s / T, torch.sigmoid(t / T)) * T * T

# ---------- self-test: shapes + adapters are the identity at init ----------
CORE_SELFTEST = True
if CORE_SELFTEST:
    ids = torch.randint(0, tokenizer.get_vocab_size(), (2, 64), device=device); lens = torch.tensor([50, 64], device=device)
    for method in ["lorma", "lora"]:
        bb = make_backbone().eval()
        with torch.no_grad(): f0 = bb.features(ids, lens)
        inject(bb, method, 8, 16)
        with torch.no_grad(): f1 = bb.features(ids, lens)
        assert f0.shape == (2, 2 * bb.d) and (f0 - f1).abs().max().item() < 1e-3, f"{method}: not identity at init"
        tr, tot = n_params(bb); print(f"[{KIND}] {method}: identity OK | trainable {tr/1e6:.2f}M of {tot/1e6:.1f}M (adapters only, no head)")
        del bb; torch.cuda.empty_cache()
    bb = unfreeze_top(make_backbone(), 4); tr, tot = n_params(bb); print(f"[{KIND}] partial top-4: trainable {tr/1e6:.2f}M of {tot/1e6:.1f}M"); del bb
    print(f"✅ core ready for KIND={KIND}")

[encdec] lorma: identity OK | trainable 0.18M of 84.1M (adapters only, no head)
[encdec] lora: identity OK | trainable 0.22M of 84.1M (adapters only, no head)
[encdec] partial top-4: trainable 18.68M of 83.9M
✅ core ready for KIND=encdec


In [ ]:
for n in ["tokenizer", "device", "HF_DATASET_REPO", "HF_TOKEN", "RRModel", "build_lm", "make_backbone", "Backbone", "inject", "KIND"]:
    print("✅" if n in globals() else "❌ MISSING", n)

✅ tokenizer
✅ device
✅ HF_DATASET_REPO
✅ HF_TOKEN
✅ RRModel
✅ build_lm
✅ make_backbone
✅ Backbone
✅ inject
✅ KIND


In [12]:
import inspect, torch
bb = make_backbone(); lm = bb.lm; cls = type(lm)
print("class:", cls.__name__, "| max_src_len / max_tgt_len:", getattr(lm, "max_src_len", None), getattr(lm, "max_tgt_len", None))
print("children:", [n for n, _ in lm.named_children()])
print("position params:", [(n, tuple(p.shape)) for n, p in lm.named_parameters() if "pos" in n.lower()])
print("own methods:", [m for m in dir(cls) if not m.startswith("_") and callable(getattr(cls, m)) and m not in dir(torch.nn.Module)])
for nm in ["forward", "generate", "encode", "decode"]:
    f = getattr(cls, nm, None)
    if f is not None: print(f"\n{nm}{inspect.signature(f)}\n  names used: {f.__code__.co_names}")
src = torch.randint(0, 100, (1, 64), device=device); tgt = torch.randint(0, 100, (1, 32), device=device)
with torch.no_grad():
    try:
        o = lm(src, tgt); print("\nlm(src, tgt) ->", {k: (tuple(v.shape) if torch.is_tensor(v) else type(v).__name__) for k, v in o.items()} if isinstance(o, dict) else type(o).__name__)
    except Exception as e: print("\nlm(src, tgt) failed:", repr(e)[:300])

class: EncoderDecoderLM | max_src_len / max_tgt_len: 1024 320
children: ['token_embedding', 'enc_layers', 'enc_norm', 'dec_layers', 'dec_norm', 'lm_head']
position params: []
own methods: ['decode', 'encode', 'init_cross_kv']

forward(self, src_ids, tgt_ids)
  names used: ('size', 'max_tgt_len', 'encode', 'init_cross_kv', 'torch', 'full', 'bos_id', 'dtype', 'device', 'cat', 'decode', 'F', 'cross_entropy', 'reshape', 'vocab_size')

encode(self, src_ids)
  names used: ('size', 'max_src_len', 'token_embedding', 'enc_layers', 'cos', 'sin', 'enc_norm')

decode(self, dec_ids, cross_kv, past_key_values=None)
  names used: ('token_embedding', 'enumerate', 'dec_layers', 'cos', 'sin', 'append', 'lm_head', 'dec_norm', 'tuple')

lm(src, tgt) -> {'loss': (), 'logits': (1, 32, 32000)}


In [17]:
import os, time, json, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
try: from rouge_score import rouge_scorer
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge_score"]); from rouge_score import rouge_scorer
assert KIND == "encdec", "this cell is for the enc-dec notebook"
scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
R6 = globals().get("R6", {}); BS, SUM_MAX = 8, 320
TRAIN = [r for r in D6["train"] if r["n"] >= 1024]; VAL = [r for r in D6["val"] if r["n"] >= 1024]
print("train docs used:", len(TRAIN), "of", len(D6["train"]), "| val:", len(VAL), "of", len(D6["val"]))

def make_src(r): return doc_view(r, 768, 256).astype(np.int64).tolist()
def make_tgt(r):
    if r["s_len"] <= SUM_MAX - 1: return r["summ"][:r["s_len"]].astype(np.int64).tolist() + [EOS6]
    return r["summ"][:SUM_MAX].astype(np.int64).tolist()
def batchify_ed(rs):
    src = torch.tensor([make_src(r) for r in rs], device=device); ts = [make_tgt(r) for r in rs]; T = max(map(len, ts))
    tgt = torch.zeros(len(rs), T, dtype=torch.long); lab = torch.full((len(rs), T), -100, dtype=torch.long)
    for i, t in enumerate(ts): tgt[i, :len(t)] = torch.tensor(t); lab[i, :len(t)] = torch.tensor(t)
    return src, tgt.to(device), lab.to(device)
def ed_loss(lm, src, tgt, lab):
    lg = lm(src, tgt)["logits"]; return F.cross_entropy(lg.float().reshape(-1, lg.size(-1)), lab.reshape(-1), ignore_index=-100)

@torch.no_grad()
def val_loss_ed(bb, n=None):
    bb.eval(); vs = VAL[:n] if n else VAL; out = []
    for i in range(0, len(vs), BS):
        src, tgt, lab = batchify_ed(vs[i:i + BS])
        with torch.autocast("cuda", dtype=torch.bfloat16): out.append(ed_loss(bb.lm, src, tgt, lab).item())
    bb.train(); return float(np.mean(out))

def _dec(lm, ids, ckv, past=None):
    o = lm.decode(ids, ckv, past_key_values=past)
    if isinstance(o, dict): return o["logits"], o.get("past_key_values")
    if isinstance(o, (tuple, list)): return o[0], (o[1] if len(o) > 1 else None)
    return o, None

@torch.no_grad()
def generate_ed(bb, src, max_new=SUM_MAX, nrep=4):
    lm = bb.lm.eval(); out, seen = [], {}
    with torch.autocast("cuda", dtype=torch.bfloat16):
        ckv = lm.init_cross_kv(lm.encode(torch.tensor([src], device=device)))
        cur = torch.full((1, 1), lm.bos_id, device=device); past = None
        for _ in range(max_new):
            lg, past = _dec(lm, cur, ckv, past); lg = lg[0, -1].float()
            key = tuple(out[-(nrep - 1):]) if nrep and len(out) >= nrep - 1 else None
            if key is not None and key in seen: lg[list(seen[key])] = -float("inf")
            t = int(lg.argmax())
            if t == EOS6: break
            if key is not None: seen.setdefault(key, set()).add(t)
            out.append(t); nxt = torch.tensor([[t]], device=device)
            cur = nxt if past is not None else torch.cat([cur, nxt], 1)      # no cache -> recompute the prefix
    return out

def rouge_eval_ed(bb, n=40, max_new=SUM_MAX, show=0):
    r1, rl, wl = [], [], []
    for j, r in enumerate(VAL[:n]):
        hyp = tokenizer.decode(generate_ed(bb, make_src(r), max_new)); ref = tokenizer.decode(r["summ"].astype(np.int64).tolist())
        sc = scorer.score(ref, hyp); r1.append(sc["rouge1"].fmeasure); rl.append(sc["rougeL"].fmeasure); wl.append(len(hyp.split()))
        if j < show: print(f"\n--- val doc {j} ---\nHYP: {hyp[:600]}\nREF: {ref[:600]}")
    return dict(rouge1=float(np.mean(r1)), rougeL=float(np.mean(rl)), words=float(np.mean(wl)))

def train_ed(bb, name, lr, epochs=3, wd=0.01, patience=3, debug=False):
    params = [p for p in bb.parameters() if p.requires_grad]; assert params, "nothing trainable"
    tr = TRAIN[:64] if debug else TRAIN; epochs = 1 if debug else epochs
    steps = epochs * (len(tr) // BS); warm = max(1, int(0.06 * steps)); ev = max(1, steps // (2 * epochs))
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    path = f"{P6_DIR}/sm_{name}_best.pt"; best, bad, step, run, t0, stop = 1e9, 0, 0, [], time.time(), False
    ntr, ntot = n_params(bb); print(f"[{name}] trainable {ntr/1e6:.2f}M / {ntot/1e6:.1f}M | steps {steps} | eval every {ev}")
    torch.cuda.reset_peak_memory_stats(); bb.train()
    for ep in range(epochs):
        order = np.random.RandomState(ep).permutation(len(tr))
        for k in range(0, len(tr) - BS + 1, BS):
            if step >= steps or stop: break
            src, tgt, lab = batchify_ed([tr[i] for i in order[k:k + BS]])
            with torch.autocast("cuda", dtype=torch.bfloat16): loss = ed_loss(bb.lm, src, tgt, lab)
            loss.backward(); run.append(loss.item())
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
            if step % ev == 0 or step == steps:
                vl = val_loss_ed(bb, 16 if debug else None)
                print(f"step {step}/{steps} | train {np.mean(run):.3f} | val {vl:.3f} | {(time.time()-t0)/60:.1f} min"); run = []
                if vl < best:
                    best, bad = vl, 0; torch.save({n: p.detach().cpu() for n, p in bb.named_parameters() if p.requires_grad}, path)
                else:
                    bad += 1; stop = bad >= patience
    bb.load_state_dict(torch.load(path), strict=False)
    m = rouge_eval_ed(bb, n=6 if debug else 40, show=1 if debug else 0)
    R6[name] = dict(val_loss=best, **m, trainable_M=ntr / 1e6, peak_gb=torch.cuda.max_memory_allocated() / 1e9, minutes=(time.time() - t0) / 60)
    print(f"\n✅ [{name}] best val loss {best:.3f} | ROUGE-1 {m['rouge1']:.4f} | ROUGE-L {m['rougeL']:.4f} | avg words {m['words']:.0f} | peak {R6[name]['peak_gb']:.1f} GB")
    return bb

def write_pred_ed(bb, name):
    preds = []; t0 = time.time()
    for i, r in enumerate(D6["test"]):
        h = tokenizer.decode(generate_ed(bb, make_src(r))).strip()
        preds.append(dict(id=r["id"], summary=h or tokenizer.decode(doc_view(r, 150).astype(np.int64).tolist())))
        if (i + 1) % 25 == 0: print(f"{i+1}/100 | {(time.time()-t0)/60:.1f} min")
    assert len(preds) == 100 and len({p["id"] for p in preds}) == 100
    p = f"{P6_DIR}/prediction_{name}.json"; json.dump(preds, open(p, "w")); print("✅ wrote", p, "-> rename to prediction.json, zip, upload")

def inject_ed(bb, method, r, alpha):
    """LoRA / LoRMA+ on q,k,v,out projections of the encoder AND decoder (self- and cross-attention)."""
    for p in bb.lm.parameters(): p.requires_grad_(False)
    cls = {"lora": LoRALinear, "lorma": LoRMAPlusLinear}[method]; cnt = {}
    for name, mod in list(bb.lm.named_modules()):
        for cn, child in list(mod.named_children()):
            full = f"{name}.{cn}" if name else cn
            if isinstance(child, nn.Linear) and cn in TARGETS["encdec"] and full.split(".")[0] in ("enc_layers", "dec_layers"):
                setattr(mod, cn, cls(child, r, alpha).to(child.weight.device)); cnt[full.split(".")[0] + "." + cn] = cnt.get(full.split(".")[0] + "." + cn, 0) + 1
    print("adapters:", cnt); return bb

def selfcheck_ed():
    bb = make_backbone().eval(); lm = bb.lm; r = VAL[0]
    src = torch.tensor([make_src(r)], device=device); tgt = torch.randint(1, 1000, (1, 12), device=device)
    print("init_cross_kv", __import__("inspect").signature(lm.init_cross_kv), "| decoder-layer-0 linears:", [n for n, m in lm.dec_layers[0].named_modules() if isinstance(m, nn.Linear)])
    with torch.no_grad():
        o = lm(src, tgt); full = o["logits"][0].float()
        man = F.cross_entropy(full, tgt[0]); print("model loss %.4f | manual CE %.4f" % (o["loss"].item(), man.item()))
        ckv = lm.init_cross_kv(lm.encode(src)); cur = torch.full((1, 1), lm.bos_id, device=device); past = None; worst = 0.0
        for t in range(12):
            lg, past = _dec(lm, cur, ckv, past); worst = max(worst, (lg[0, -1].float() - full[t]).abs().max().item())
            cur = tgt[:, t:t + 1] if past is not None else torch.cat([cur, tgt[:, t:t + 1]], 1)
    print("cache returned:", type(past).__name__, "| max diff cached vs teacher-forced: %.2e" % worst); del bb
selfcheck_ed(); print("✅ enc-dec toolkit ready")

train docs used: 6507 of 6728 | val: 296 of 300
init_cross_kv (enc_out) | decoder-layer-0 linears: ['self_attn.q_proj', 'self_attn.k_proj', 'self_attn.v_proj', 'self_attn.out_proj', 'cross_attn.q_proj', 'cross_attn.k_proj', 'cross_attn.v_proj', 'cross_attn.out_proj', 'ffn.w1', 'ffn.w2', 'ffn.w3']
model loss 12.0705 | manual CE 12.0705
cache returned: tuple | max diff cached vs teacher-forced: 2.29e-05
✅ enc-dec toolkit ready


In [ ]:
bb = make_backbone(); train_ed(bb, "dbg_ed", lr=5e-5, debug=False); bb = None; torch.cuda.empty_cache()

[dbg_ed] trainable 83.91M / 83.9M | steps 2439 | eval every 406
step 406/2439 | train 3.023 | val 2.890 | 0.7 min
step 812/2439 | train 2.894 | val 2.820 | 1.4 min
step 1218/2439 | train 2.692 | val 2.797 | 2.1 min
step 1624/2439 | train 2.675 | val 2.773 | 2.7 min
step 2030/2439 | train 2.530 | val 2.773 | 3.4 min
step 2436/2439 | train 2.543 | val 2.767 | 4.1 min
step 2439/2439 | train 2.615 | val 2.767 | 4.1 min

✅ [dbg_ed] best val loss 2.767 | ROUGE-1 0.3727 | ROUGE-L 0.2125 | avg words 272 | peak 10.0 GB


In [ ]:
bb = make_backbone(); bb = train_ed(bb, "encdec_full", lr=5e-5)
write_pred_ed(bb, "encdec_full"); bb = None; torch.cuda.empty_cache()

[encdec_full] trainable 83.91M / 83.9M | steps 2439 | eval every 406
step 406/2439 | train 3.023 | val 2.890 | 0.7 min
step 812/2439 | train 2.894 | val 2.820 | 1.4 min
step 1218/2439 | train 2.692 | val 2.797 | 2.0 min
step 1624/2439 | train 2.675 | val 2.773 | 2.7 min
step 2030/2439 | train 2.530 | val 2.773 | 3.4 min
step 2436/2439 | train 2.543 | val 2.767 | 4.1 min
step 2439/2439 | train 2.615 | val 2.767 | 4.1 min

✅ [encdec_full] best val loss 2.767 | ROUGE-1 0.3727 | ROUGE-L 0.2125 | avg words 272 | peak 10.0 GB
25/100 | 1.5 min
50/100 | 2.9 min
75/100 | 4.4 min
100/100 | 5.9 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_6/prediction_encdec_full.json -> rename to prediction.json, zip, upload
